In [0]:
import sys
import numpy as np
import pandas as pd

# sys.path.append()
from contract_dates_functions import *

# sys.path.append()
from implied_cnh_rate_functions import *


## Importing and Loading Data (standard)

In [0]:
# # non settlement dates
# fed_holidays = set(
#     pd.to_datetime(
#         pd.read_csv('FD_holidays_data.csv').iloc[:, 0], 
#         dayfirst=True
#     )
# )

# cnh_holidays = set(
#     pd.to_datetime(
#         pd.read_csv('Cplus_holidays_data.csv').iloc[:, 0], 
#         dayfirst=True
#     )
# )

# # expiry dates
# expiry_df = pd.read_csv(
#     'Expiry_dates.csv',
#     parse_dates=['Trading date'] + tenor_cols,  # Parse ALL date columns
#     dayfirst=True,
#     index_col='Trading date'
# )

## Reviewing maturity dates

In [0]:
# # REVIEWING MATURITY DATES

# # Load forward contract maturity dates
# maturity_dates = pd.read_csv('', 
#                               parse_dates=['Maturity'])

# maturity_date_list = maturity_dates['Maturity'].tolist()

# # Analyse original maturity dates (before augmentation)
# maturity_df = pd.DataFrame({'Maturity': maturity_date_list})
# maturity_df['Weekday'] = pd.to_datetime(maturity_df['Maturity']).dt.day_name()
# maturity_df['Day'] = pd.to_datetime(maturity_df['Maturity']).dt.day

# # Count by weekday
# weekday_counts = maturity_df['Weekday'].value_counts()
# print("Maturity dates by weekday:")
# print(weekday_counts)

# # Average day of month
# avg_day = maturity_df['Day'].mean()
# print(f"\nAverage day of month: {avg_day:.1f}")

# print(f"\nAverage 21st of the month / thursday or wednesday")

# print(f"\nFor liquidity concerns in December and to not overfit the data, will do consisent mid month")

# print(f"\nFirst maturity date was {maturity_date_list[0]}")

## Creating the csv of all contract dates for 3 positions

1. Create synthetic maturity dates

In [0]:
# SETTING UP BACKTESTER MATURITY DATES

# Generate all (specific weekday e.g. wednesday)
specific_weekdays = pd.date_range(start='2021-04-01', end='2026-07-30', freq='W-WED')

# Group by year-month and take index of the month
synthetic_maturity_list = (
    specific_weekdays.to_series()
    .groupby([specific_weekdays.year, specific_weekdays.month])
    .nth(-3) # -3 = third last
    .tolist()
)

# Adjust for holidays and track changes
adjusted_list = []
changed_dates = []

for original_date in synthetic_maturity_list:
    date = original_date
    while date.weekday() >= 5 or date in FED_HOLIDAYS or date in CNH_HOLIDAYS:
        date = date + pd.Timedelta(days=1)
    
    adjusted_list.append(date)
    
    # Track if date was changed
    if date != original_date:
        changed_dates.append({
            'Original': original_date,
            'Adjusted': date,
            'Weekday': date.strftime('%A'),
            'Reason': 'fed holiday' if original_date in FED_HOLIDAYS else 'cnh holiday'
        })

synthetic_maturity_list = adjusted_list

print(f"Total dates: {len(synthetic_maturity_list)}")
print(f"First: {synthetic_maturity_list[0]}")
print(f"Last: {synthetic_maturity_list[-1]}")

# Print changed dates
if changed_dates:
    print(f"\n{len(changed_dates)} dates were adjusted:")
    for change in changed_dates:
        print(f"  {change['Original'].strftime('%Y-%m-%d')} → {change['Adjusted'].strftime('%Y-%m-%d')} ({change['Weekday']}) ({change['Reason']})")
else:
    print("\nNo dates were adjusted")


Total dates: 64
First: 2021-04-14 00:00:00
Last: 2026-07-15 00:00:00

2 dates were adjusted:
  2024-02-14 → 2024-02-20 (Tuesday) (cnh holiday)
  2024-05-15 → 2024-05-16 (Thursday) (cnh holiday)


2. Find matching expiry dates and exit dates

In [0]:
# Choose exit window
minimum_days_before_expiry = 7

# Step 1: Extract all unique expiry dates from the DataFrame (excluding index)
unique_expiry_dates = pd.Series(expiry_df.values.flatten()).dropna().unique()
unique_expiry_dates = pd.to_datetime(unique_expiry_dates)
print(f"Total unique expiry dates: {len(unique_expiry_dates)}")

# Step 2: Create (expiry_date, delivery_date) pairs for all of the unique expiry dates
unique_expiry_delivery_pairs = []
for expiry_date in unique_expiry_dates:
    delivery_date = get_T2_lag_date(expiry_date)
    unique_expiry_delivery_pairs.append((expiry_date, delivery_date))

print(f"Total expiry-delivery pairs: {len(unique_expiry_delivery_pairs)}")

# Step 3: Match synthetic maturity dates to delivery dates
synthetic_expiry_delivery_pairs = []
for maturity_date in synthetic_maturity_list:
    # Find matching pair where delivery_date == maturity_date
    matching_pair = None
    for expiry_date, delivery_date in unique_expiry_delivery_pairs:
        if delivery_date == maturity_date:
            matching_pair = (expiry_date, delivery_date)
            break
    
    if matching_pair is None:
        print(f"No matching pair for {maturity_date.strftime('%Y-%m-%d')}")
    else:
        synthetic_expiry_delivery_pairs.append(matching_pair)

# Step 4: Create (expiry, delivery, exit_date) triples
synthetic_expiry_delivery_exit_triples = []
for expiry_date, delivery_date in synthetic_expiry_delivery_pairs:
    # Calculate exit date
    exit_date = expiry_date - pd.Timedelta(days=minimum_days_before_expiry)
    
    # Adjust backward to Friday if lands on weekend
    while exit_date.weekday() >= 5:
        exit_date = exit_date - pd.Timedelta(days=1)

    # adjust if exit settlement would be on/after delivery (for pricing reasons - implied rate)
    while get_T2_lag_date(exit_date) >= delivery_date:
        exit_date = exit_date - pd.Timedelta(days=1)
    synthetic_expiry_delivery_exit_triples.append((expiry_date, delivery_date, exit_date))

print(f"\nTotal synthetic expiry-delivery-exit triples: {len(synthetic_expiry_delivery_exit_triples)}")
print(f"First triple: {synthetic_expiry_delivery_exit_triples[0] if synthetic_expiry_delivery_exit_triples else 'None'}")
print(f"Last triple: {synthetic_expiry_delivery_exit_triples[-1] if synthetic_expiry_delivery_exit_triples else 'None'}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6912466321754940>, line 5
      2 minimum_days_before_expiry = 7
      4 # Step 1: Extract all unique expiry dates from the DataFrame (excluding index)
----> 5 unique_expiry_dates = pd.Series(expiry_df.values.flatten()).dropna().unique()
      6 unique_expiry_dates = pd.to_datetime(unique_expiry_dates)
      7 print(f"Total unique expiry dates: {len(unique_expiry_dates)}")

NameError: name 'expiry_df' is not defined

3. set up all contract dates for 3 staggered positions

In [0]:
def get_expiry_exit_from_delivery(delivery_date, expiry_delivery_exit_triples):
    """Lookup expiry and exit dates from delivery date using pre-calculated triples"""
    for expiry, delivery, exit_date in expiry_delivery_exit_triples:
        if delivery == delivery_date:
            return expiry, exit_date
    raise ValueError(f"No expiry/exit dates found for delivery date {delivery_date.strftime('%Y-%m-%d')}")

# on first delivery date we start the strategy 
start_date = synthetic_maturity_list[0]
print(start_date)

# position starts
# set up
NUM_POSITIONS = 3
MATURITY_INCREMENT = 3  # roll to maturity date n+3 (3 months)
CUTOFF_DATE = pd.Timestamp('2026-07-30')
  
# initialise tracking
position_contracts = {pos: [] for pos in range(NUM_POSITIONS)}

# initialise positions with maturity dates 2, 3, 4
current_maturity_indices = {}

for pos in range(NUM_POSITIONS):
    maturity_index = pos + 1  # positions 0,1,2 get maturity dates 1,2,3 to start
    current_maturity_indices[pos] = maturity_index
    
    # Set initial start date
    current_start_date = start_date
    
    # Loop to generate all contracts for this position
    while True:
        # Check if current start date is after cutoff - if so, stop this position
        if current_start_date > CUTOFF_DATE:
            break
        
        # Check if next maturity index exists
        if current_maturity_indices[pos] >= len(synthetic_maturity_list):
            break
        
        # Get delivery date for current maturity index
        delivery_date = synthetic_maturity_list[current_maturity_indices[pos]]
        
        # Get expiry and exit dates from triples
        expiry_date, exit_date = get_expiry_exit_from_delivery(
            delivery_date, 
            synthetic_expiry_delivery_exit_triples
        )
        
        # Calculate settlement date
        settlement_date = get_T2_lag_date(current_start_date)
        
        # # Verify dates
        # contract_dates_check(current_start_date, settlement_date, expiry_date, delivery_date)
        
        # Store contract
        position_contracts[pos].append({
            'start_date': current_start_date,
            'settlement_date': settlement_date,
            'delivery_date': delivery_date,
            'expiry_date': expiry_date,
            'exit_date': exit_date,
        })
        
        # Move to next contract
        current_maturity_indices[pos] += MATURITY_INCREMENT
        
        # Next contract starts on exit date of current contract
        current_start_date = exit_date

# Print summary
print(f"\nStrategy completed. Total contracts per position:")
for pos in range(NUM_POSITIONS):
    print(f"Position {pos+1}: {len(position_contracts[pos])} contracts")

# Flatten results
all_contracts = []
contract_index = 0
for pos in range(NUM_POSITIONS):
    for contract in position_contracts[pos]:
        contract_index += 1
        all_contracts.append({
            'Contract_ID': contract_index,
            'Position': pos + 1,
            'Start_Date': contract['start_date'],
            'Settlement_Date': contract['settlement_date'],
            'Exit_Date': contract['exit_date'],
            'Expiry_Date': contract['expiry_date'],
            'Delivery_Date': contract['delivery_date']
        })

# Save to csv
contracts_df = pd.DataFrame(all_contracts)
contracts_df.to_csv('contract_dates.csv', index=False)

Check they have tenors (for vol stuff)

In [0]:
for idx, contract in contracts_df.iterrows():
    contract_id = contract['Contract_ID']
    position = contract['Position']
    start_date = contract['Start_Date']
    settlement_date = contract['Settlement_Date']
    exit_date = contract['Exit_Date']
    expiry_date = contract['Expiry_Date']
    delivery_date = contract['Delivery_Date']

    try:
      tenor = get_tenor_from_expiry(start_date, expiry_date)
    except ValueError:
      print(f'\n no tenor for Contract {contract_id}')
      print(f'\n start {start_date}')
      print(f'\n expiry {expiry_date}')
      print(f'\n trade to expiry {(expiry_date - start_date).days}')
    
    if tenor is not None:
      if not settlement_date, expiry_date, delivery_date == get_contract_dates(start_date, tenor):
        print(f'\n contract dates dont match Contract {contract_id}')
        print(f'\n tenor {tenor}')
        print(f'\n start {start_date}')
        print(f'\n sett {settlement_date}')
        print(f'\n expiry {expiry_date}')
        print(f'\n expiry {delivery_date}')
      

Test if contract dates and tenor match

In [0]:
settlement_date, expiry_date, delivery_date == get_contract_dates(trading_date, tenor)